# Упражнение: Трекинг экспериментов: версия данных по хешу и выбор лучшего запуска

## Задание

В `runs.jsonl` — журнал экспериментов (по одному JSON-объекту на строку, как в лекции): `run_id`, `status`,
`params`, `metrics` (`cv_auc`, `cv_auc_std`, `test_auc`), `data_version`, `git_commit`.
В `input.csv` — **текущая** версия обучающих данных. Часть запусков сделана на старой версии.

1. Вычислите версию текущих данных так же, как её записывал трекер: прочитайте `input.csv` как текст
   (`Path('input.csv').read_text(encoding='utf-8')` — переводы строк при этом приводятся к `\n`),
   возьмите MD5 от этого текста в UTF-8 и первые 8 шестнадцатеричных символов.
2. `data_ok` = 1, если `data_version` запуска совпадает с текущей версией.
3. `eligible` = 1, если `data_ok = 1` и `status == 'FINISHED'`.
4. Среди допустимых запусков отранжируйте по `cv_auc` по убыванию; при равенстве выше тот,
   у кого меньше `cv_auc_std`, затем — меньший `run_id`. `rank` — место (1, 2, …), у недопустимых 0.
5. `best` = 1 у запуска с `rank = 1`. `test_auc` для выбора **не используется**: отложенная выборка
   нужна для финальной оценки, а не для отбора.

## Подсказки

- Журнал читается так: `[json.loads(line) for line in Path('runs.jsonl').read_text(encoding='utf-8').splitlines()]`,
  а `pd.json_normalize(runs)` развернёт вложенные `metrics` в столбцы `metrics.cv_auc` и т.д.
- Хеш: `hashlib.md5(text.encode('utf-8')).hexdigest()[:8]`. Хеш от байтов файла (`read_bytes`) может
  отличаться из-за переводов строк Windows — поэтому трекер хеширует нормализованный текст.
- Ранжирование с несколькими ключами: `sort_values(['cv_auc', 'cv_auc_std', 'run_id'], ascending=[False, True, True])`
  по допустимым запускам, затем `rank = 1..k` по полученному порядку.

## Ожидаемый формат выходного файла

CSV с колонками `run_id, data_ok, eligible, rank, best` — по строке на каждый запуск в порядке журнала;
все значения, кроме `run_id`, целые.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/mlops_best_practices
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.